<a href="https://colab.research.google.com/github/Mustafa-karcak/GROUP56_CENG313/blob/main/assignment1_eda_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import files
uploaded = files.upload()

Saving kaggle.json to kaggle (1).json


In [6]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c drw-crypto-market-prediction -p data
!unzip -q -o data/drw-crypto-market-prediction.zip -d data
!ls -lh data

100% 5.98G/5.98G [01:10<00:00, 91.5MB/s]

total 13G
-rw-r--r-- 1 root root 6.0G Jul  9  2025 drw-crypto-market-prediction.zip
-rw-r--r-- 1 root root  14M Jul  9  2025 sample_submission.csv
-rw-r--r-- 1 root root 3.2G Jul  9  2025 test.parquet
-rw-r--r-- 1 root root 3.1G Jul  9  2025 train.parquet


In [7]:
!git clone https://github.com/Mustafa-karcak/GROUP56_CENG313.git
import sys
sys.path.append('GROUP56_CENG313/src')

Cloning into 'GROUP56_CENG313'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 41 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 21.95 KiB | 3.13 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [8]:
!ls GROUP56_CENG313/src
!head -5 GROUP56_CENG313/src/data.py

data.py  models.py  utils.py
"""Data loading helpers for DRW Crypto Market Prediction (CENG313 Assignment 1)."""
import numpy as np
import pandas as pd

MARKET_COLS = ['bid_qty', 'ask_qty', 'buy_qty', 'sell_qty', 'volume']


In [9]:
!cd GROUP56_CENG313 && git pull

import sys
sys.path.insert(0, 'GROUP56_CENG313/src')
for m in ['data', 'utils', 'models']:
    sys.modules.pop(m, None)

import data, utils, models
print(data.__file__)
print(hasattr(data, 'load_data'))   # True olmalı

Already up to date.
/content/GROUP56_CENG313/src/data.py
True


In [10]:
train = data.load_data('data/train.parquet')
print(train.shape)
print(f"{utils.memory_mb(train):.0f} MB")
print(len(data.anonymous_cols(train)), 'anonymous X features')
print(train.index[:5])
train.head()

(525886, 786)
1581 MB
780 anonymous X features
DatetimeIndex(['2023-03-01 00:00:00', '2023-03-01 00:01:00',
               '2023-03-01 00:02:00', '2023-03-01 00:03:00',
               '2023-03-01 00:04:00'],
              dtype='datetime64[ns]', freq=None)


,bid_qty,ask_qty,buy_qty,sell_qty,volume,X1,X2,X3,X4,X5,...,X772,X773,X774,X775,X776,X777,X778,X779,X780,label
2023-03-01 00:00:00,15.283000,8.425000,176.404999,44.984001,221.389008,0.181844,-0.637860,0.006652,0.136870,0.116698,...,0.333753,-0.009992,-0.695594,-0.444077,-0.191238,-0.184251,-0.471897,-0.625428,-0.553991,0.562539
2023-03-01 00:01:00,38.590000,2.336000,525.846008,321.950012,847.796021,0.489497,-0.075619,0.431594,0.522400,0.475255,...,0.333658,-0.010040,-0.696226,-0.452866,-0.200082,-0.188929,-0.472842,-0.625832,-0.554426,0.533686
2023-03-01 00:02:00,0.442000,60.250000,159.227005,136.369003,295.596008,0.260121,-0.444684,0.100695,0.224729,0.203282,...,0.333667,-0.010037,-0.696832,-0.461383,-0.208786,-0.193571,-0.473785,-0.626236,-0.554860,0.546505
2023-03-01 00:03:00,4.865000,21.016001,335.742004,124.962997,460.704987,0.099976,-0.666728,-0.123858,0.019197,0.014459,...,0.333174,-0.010279,-0.697391,-0.469628,-0.217350,-0.198175,-0.474726,-0.626639,-0.555294,0.357703
2023-03-01 00:04:00,27.158001,3.451000,98.411003,44.407001,142.817993,0.270893,-0.325973,0.116336,0.234311,0.214073,...,0.333171,-0.010283,-0.697940,-0.477622,-0.225780,-0.202745,-0.475666,-0.627043,-0.555728,0.362452


In [11]:
print("Missing values:")
print(utils.missing_report(train).head(20))

print("\nInfinite values:")
print(utils.infinite_report(train).head(20))
train = data.clean_infinite(train)

const = utils.constant_columns(train)
print(f"\n{len(const)} constant columns:", const[:10])

print('\nduplicate timestamps:', train.index.duplicated().sum())
print('index sorted by time:', train.index.is_monotonic_increasing)

Missing values:
Empty DataFrame
Columns: [missing, percent]
Index: []

Infinite values:
Series([], dtype: int64)

0 constant columns: []

duplicate timestamps: 0
index sorted by time: True


Data quality: No missing values, no infinite values, no constant columns, no duplicate timestamps; the index is sorted in time. No cleaning needed.

In [12]:
mk = data.MARKET_COLS
print("Negative values per market feature:")
print((train[mk] < 0).sum())

gap = (train['buy_qty'] + train['sell_qty'] - train['volume']).abs()
print("\n|buy_qty + sell_qty - volume|:")
print(gap.describe())

Negative values per market feature:
bid_qty     0
ask_qty     0
buy_qty     0
sell_qty    0
volume      0
dtype: int64

|buy_qty + sell_qty - volume|:
count    525886.000000
mean          0.000005
std           0.000026
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.001953
dtype: float64


Sniff tests: No negative quantities. volume = buy_qty + sell_qty in every row (max difference 0.002, float rounding), so volume adds no new information.